# Aksara Jawa Image Preprocessing

This notebook handles the data acquisition and preprocessing pipeline for the Aksara Jawa (Javanese script) dataset. The pipeline ensures that all images are uniform, clean, and ready to be fed into a computer vision model.

## 1. Download Dataset

First, we download the dataset from Kaggle using the `kagglehub` library. The dataset will be downloaded and safely extracted into the `./data` directory.


In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("phiard/aksara-jawa", output_dir="./data")

print("Path to dataset files:", path)

Path to dataset files: ./data


## 2. Image Preprocessing Pipeline

The downloaded images may have varying sizes and irregular white borders. This step standardizes the dataset by applying the following optimized operations:

1. **Automatic Cropping (`automatic_crop`)**:
   - **Dynamic Thresholding**: Uses Otsu's Binarization together with inverted binary thresholding to dynamically separate text from background.
   - **Dynamic Cropping Margins**: Adds an adaptive margin (5% of the largest dimension) to scale dynamically based on image size.
   - **Diacritic-Safe Bounding**: Finds coordinates using `findNonZero` to safely preserve detached Javanese diacritics (sandhangan) without aggressive morphological noise removal.
2. **Resize and Pad (`resize_and_pad`)**:
   - Uses **Smart Interpolation** (`cv2.INTER_AREA` for downscaling, `cv2.INTER_CUBIC` for upscaling) to preserve the sharpness of the structural lines of the characters.
   - Adds pure white padding (letterboxing) evenly to ensure a perfect square shape without distorting their original aspect ratio.
3. **Parallel Batch Processing (`process_complete_dataset`)**:
   - Employs **Multiprocessing** via `concurrent.futures.ProcessPoolExecutor` to process individual images in parallel, fully utilizing multiple CPU cores for speed improvements.
   - Saves processed copies securely as **Lossless Output** in `.png` format to completely avert JPEG compression artifacts moving forward.


In [2]:
import cv2
import os
import numpy as np
from pathlib import Path
import concurrent.futures


# --- FUNGSI 1: Automatic Cropping ---
def automatic_crop(img):
    """
    Melakukan crop otomatis untuk menghilangkan background putih di sekitar aksara.
    """
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 1. Dynamic Thresholding: Otsu's Binarization
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)

    # Menggunakan findNonZero agar semua partikel tulisan (termasuk sandhangan) terbaca
    coords = cv2.findNonZero(thresh)

    if coords is None:
        return img

    x, y, w, h = cv2.boundingRect(coords)

    # 6. Edge Case Handling: Abaikan noise atau bounding box yang terlalu kecil
    if w < 5 or h < 5:
        return img

    # 2. Dynamic Cropping Margins: 1,618% dari sisi terpanjang gambar
    margin = int(0.01618 * max(img.shape[0], img.shape[1]))

    # Tambahkan margin dengan batasan ukuran gambar agar tidak out-of-bounds
    y_start = max(0, y - margin)
    y_end = min(img.shape[0], y + h + margin)
    x_start = max(0, x - margin)
    x_end = min(img.shape[1], x + w + margin)

    cropped_img = img[y_start:y_end, x_start:x_end]

    return cropped_img


# --- FUNGSI 2: Resize dengan Padding (Letterboxing) ---
def resize_and_pad(img, target_size=128):
    """
    Resize gambar tanpa distorsi (menjaga aspect ratio)
    dan menambahkan padding putih agar ukuran menjadi persegi.
    """
    old_size = img.shape[:2]
    ratio = float(target_size) / max(old_size)
    new_size = tuple([int(x * ratio) for x in old_size])

    # 3. Smart Interpolation
    if ratio < 1.0:
        interpolation = cv2.INTER_AREA  # Untuk Downscaling
    else:
        interpolation = cv2.INTER_CUBIC  # Untuk Upscaling

    img_resized = cv2.resize(
        img, (new_size[1], new_size[0]), interpolation=interpolation
    )

    # Hitung dan terapkan padding putih
    delta_w = target_size - new_size[1]
    delta_h = target_size - new_size[0]

    top, bottom = delta_h // 2, delta_h - (delta_h // 2)
    left, right = delta_w // 2, delta_w - (delta_w // 2)

    color = [255, 255, 255]
    final_img = cv2.copyMakeBorder(
        img_resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=color
    )

    return final_img


# --- FUNGSI HELPER PARALEL ---
def process_single_image(args):
    """Memproses satu gambar (dipakai dalam ProcessPoolExecutor)"""
    img_path, class_output_path, target_size = args
    if img_path.suffix.lower() not in [".png", ".jpg", ".jpeg"]:
        return 0, 0

    img = cv2.imread(str(img_path))
    if img is None:
        return 0, 1  # 0 sukses, 1 error

    img_cropped = automatic_crop(img)
    final_img = resize_and_pad(img_cropped, target_size=target_size)

    # 5. Lossless Output: Memastikan hasil simpan menggunakan .png
    save_path = class_output_path / f"{img_path.stem}.png"
    cv2.imwrite(str(save_path), final_img)

    return 1, 0  # 1 sukses, 0 error


# --- FUNGSI UTAMA: Pipeline Pemrosesan Dataset ---
def process_complete_dataset(input_base_dir, output_base_dir, target_size=128):
    """
    Pipeline paralel untuk membaca, crop, padding+resize, dan export lossless.
    """
    input_base = Path(input_base_dir)
    output_base = Path(output_base_dir)
    sub_dirs = ["train", "val"]

    total_processed = 0
    total_errors = 0

    print("=" * 60)
    print(f"🚀 Memulai Pemrosesan Dataset Aksara Jawa (Optimized)")
    print(f"Target Ukuran: {target_size}x{target_size} px | Mode: Lossless PNG")
    print("=" * 60)

    # Daftar antrean task untuk Multiprocessing
    tasks = []

    for sub_dir in sub_dirs:
        input_path = input_base / sub_dir
        output_path = output_base / sub_dir

        if not input_path.exists():
            continue

        for class_path in sorted(input_path.iterdir()):
            if not class_path.is_dir():
                continue

            class_name = class_path.name
            class_output_path = output_path / class_name
            class_output_path.mkdir(parents=True, exist_ok=True)

            for img_path in class_path.iterdir():
                if img_path.suffix.lower() in [".png", ".jpg", ".jpeg"]:
                    tasks.append((img_path, class_output_path, target_size))

    print(f"📂 Menemukan {len(tasks)} file. Memproses secara paralel...")

    # 4. Multiprocessing: Process Pool untuk scaling ke seluruh core CPU
    with concurrent.futures.ProcessPoolExecutor() as executor:
        futures = [executor.submit(process_single_image, task) for task in tasks]

        for future in concurrent.futures.as_completed(futures):
            proc, err = future.result()
            total_processed += proc
            total_errors += err

    print("\n" + "=" * 60)
    print(f"📊 RINGKASAN PEMROSESAN:")
    print(f"   Berhasil diproses : {total_processed} gambar")
    print(f"   Gagal dibaca      : {total_errors} gambar")
    print(f"   Output di         : {output_base_dir}")
    print("=" * 60)


if __name__ == "__main__":
    # Konfigurasi Direktori & Resolusi
    INPUT_DATA_DIR = "data/v3/v3"
    OUTPUT_DATA_DIR = "data/final"
    FINAL_RESOLUTION = 224

    process_complete_dataset(INPUT_DATA_DIR, OUTPUT_DATA_DIR, FINAL_RESOLUTION)

🚀 Memulai Pemrosesan Dataset Aksara Jawa (Optimized)
Target Ukuran: 224x224 px | Mode: Lossless PNG
📂 Menemukan 2634 file. Memproses secara paralel...

📊 RINGKASAN PEMROSESAN:
   Berhasil diproses : 2634 gambar
   Gagal dibaca      : 0 gambar
   Output di         : data/final
